# Set up
Notebook setup.


In [ ]:
# Mount Google Drive so experiment inputs and outputs are available in Colab.
# Paths containing Ablation_test below are legacy Google Drive folder names, not Git branches.
# Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Clone the unified public thesis repository into the local path expected by later cells.
!rm -rf /content/NeuS_thesis
!git clone https://github.com/CowboyScarlatto42/Thesis.git /content/NeuS_thesis
!cd /content/NeuS_thesis && git branch --show-current

In [ ]:
# Install Python dependencies required by NeuS and the uncertainty scripts.
# Dipendenze principali
!pip install -q trimesh
!pip install -q opencv-python
!pip install -q plyfile
!pip install -q scipy
!pip install -q numpy

# Per visualizzazione
!pip install -q open3d
!pip install -q plotly
!pip install -q matplotlib


In [ ]:
# Install the tree utility used for quick folder inspection.
!apt-get -qq install tree

In [ ]:
# Optional folder-tree inspection command kept disabled by default.
#!tree /content/drive/MyDrive/Tesi/neus/Ablation_test/Experiments

# GPU + CONF setup
GPU and configuration setup for uncertainty experiments.


In [ ]:
# Install runtime dependencies required by NeuS training, rendering, and mesh extraction.
# Dependencies
!pip install opencv-python pyhocon icecream tqdm scipy PyMCubes trimesh

In [ ]:
# Select a balanced set of image indices from the two orbits.
import numpy as np

orbit_1 = np.rint(np.linspace(0, 60, 8)).astype(int)
orbit_2 = np.rint(np.linspace(61, 127, 8)).astype(int)

selected_indices = np.concatenate([orbit_1, orbit_2])

print("Orbit 1:", orbit_1.tolist())
print("Orbit 2:", orbit_2.tolist())
print("All:", selected_indices.tolist())

In [ ]:
# Run the deformation-grid module directly as a lightweight standalone check.
# Solo griglia
!python /content/NeuS_thesis/models/deformation_grid.py

In [ ]:
# Recompute the selected multi-orbit image indices for the final commands.
import numpy as np

orbit_1 = np.rint(np.linspace(0, 60, 8)).astype(int)
orbit_2 = np.rint(np.linspace(61, 127, 8)).astype(int)

selected_indices = np.concatenate([orbit_1, orbit_2])

print("Orbit 1:", orbit_1.tolist())
print("Orbit 2:", orbit_2.tolist())
print("All:", selected_indices.tolist())

# Final Test
Final uncertainty-proxy experiment.


In [ ]:
# Update the NeuS config paths and iteration count for the final uncertainty experiment.
# ==============================
# .conf
# ==============================

from pathlib import Path

def update_conf_params(conf_path,
                       base_exp_dir_value,
                       data_dir_value,
                       end_iter_value,
                       output_path=None):
    """
    Sostituisce nel file .conf le righe:
      base_exp_dir =
      data_dir =
      end_iter =
    con i valori passati in input.

    Se output_path è None, sovrascrive il file originale.
    """

    conf_path = Path(conf_path)
    if output_path is None:
        output_path = conf_path
    else:
        output_path = Path(output_path)

    if not conf_path.exists():
        raise FileNotFoundError(f"File .conf non trovato: {conf_path}")

    lines = conf_path.read_text(encoding="utf-8").splitlines()
    new_lines = []

    found_base = False
    found_data = False
    found_end_iter = False

    for line in lines:
        stripped = line.strip()

        if stripped.startswith("base_exp_dir"):
            new_lines.append(f"base_exp_dir = {base_exp_dir_value}")
            found_base = True

        elif stripped.startswith("data_dir"):
            new_lines.append(f"data_dir = {data_dir_value}")
            found_data = True

        elif stripped.startswith("end_iter"):
            new_lines.append(f"end_iter = {end_iter_value}")
            found_end_iter = True

        else:
            new_lines.append(line)

    if not found_base:
        raise ValueError("Nel file .conf non è stata trovata una riga che inizia con 'base_exp_dir'")
    if not found_data:
        raise ValueError("Nel file .conf non è stata trovata una riga che inizia con 'data_dir'")
    if not found_end_iter:
        raise ValueError("Nel file .conf non è stata trovata una riga che inizia con 'end_iter'")

    output_path.write_text("\n".join(new_lines) + "\n", encoding="utf-8")
    print(f"[OK] File aggiornato salvato in: {output_path}")


# =========
# ESEMPIO
# =========

conf_path = "/content/NeuS_thesis/confs/long_test.conf"
base_exp_dir_value =  "/content/drive/MyDrive/Tesi/neus/Ablation_test/Experiments/07_TO_80/training"
#
data_dir_value = "/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_COLMAP/Merged_Orbit_80/Merged_Orbit_neus"
#
end_iter_value = 300000

update_conf_params(
    conf_path=conf_path,
    base_exp_dir_value=base_exp_dir_value,
    data_dir_value=data_dir_value,
    end_iter_value=end_iter_value
)

In [ ]:
# Accumulate the multi-view geometry proxy H from selected foreground rays.
# Multi - View
%cd /content/NeuS_thesis

!python custom_codes/uncertainty/accumulate_multiview_geometry_proxy.py \
  --conf ./confs/long_test.conf \
  --image_indices 0,9,17,26,34,43,51,60,61,70,80,89,99,108,118,127 \
  --rays_per_image 64 \
  --grid_resolution 32 \
  --seed 42 \
  --output_dir /content/drive/MyDrive/Tesi/neus/Ablation_test/bayesSDF/seed_42/multiview_geometry_16views

In [ ]:
# Print the validation summary produced for U on the seed_42 run.
!cat /content/drive/MyDrive/Tesi/neus/Ablation_test/bayesSDF/seed_42/mesh_validation/spearman_summary.json

In [ ]:
# Display the sparsification curve generated by the U validation script.
from IPython.display import Image, display

display(
    Image(
        filename="/content/drive/MyDrive/Tesi/neus/Ablation_test/bayesSDF/"
        "seed_42/mesh_validation/sparsification_curve.png"
    )
)

## Miglior Caso Ablation
Additional ablation case used for comparison.


In [ ]:
# Load a trained NeuS checkpoint and export a normalized mesh for the ablation case.
%cd /content/NeuS_thesis

from exp_runner import Runner
import shutil

runner = Runner(
    "/content/NeuS_thesis/confs/long_test.conf",
    mode="validate_mesh",
    case="",
    is_continue=True,
)

runner.validate_mesh(
    world_space=False,
    resolution=512,
    threshold=0.0,
)

mesh_path = f"{runner.base_exp_dir}/meshes/{runner.iter_step:08d}.ply"
normalized_path = f"{runner.base_exp_dir}/meshes/{runner.iter_step:08d}_normalized.ply"

shutil.copy(mesh_path, normalized_path)

print(normalized_path)

In [ ]:
# Update the NeuS config paths for the ablation-case uncertainty run.
# ==============================
# .conf
# ==============================

from pathlib import Path

def update_conf_params(conf_path,
                       base_exp_dir_value,
                       data_dir_value,
                       end_iter_value,
                       output_path=None):
    """
    Sostituisce nel file .conf le righe:
      base_exp_dir =
      data_dir =
      end_iter =
    con i valori passati in input.

    Se output_path è None, sovrascrive il file originale.
    """

    conf_path = Path(conf_path)
    if output_path is None:
        output_path = conf_path
    else:
        output_path = Path(output_path)

    if not conf_path.exists():
        raise FileNotFoundError(f"File .conf non trovato: {conf_path}")

    lines = conf_path.read_text(encoding="utf-8").splitlines()
    new_lines = []

    found_base = False
    found_data = False
    found_end_iter = False

    for line in lines:
        stripped = line.strip()

        if stripped.startswith("base_exp_dir"):
            new_lines.append(f"base_exp_dir = {base_exp_dir_value}")
            found_base = True

        elif stripped.startswith("data_dir"):
            new_lines.append(f"data_dir = {data_dir_value}")
            found_data = True

        elif stripped.startswith("end_iter"):
            new_lines.append(f"end_iter = {end_iter_value}")
            found_end_iter = True

        else:
            new_lines.append(line)

    if not found_base:
        raise ValueError("Nel file .conf non è stata trovata una riga che inizia con 'base_exp_dir'")
    if not found_data:
        raise ValueError("Nel file .conf non è stata trovata una riga che inizia con 'data_dir'")
    if not found_end_iter:
        raise ValueError("Nel file .conf non è stata trovata una riga che inizia con 'end_iter'")

    output_path.write_text("\n".join(new_lines) + "\n", encoding="utf-8")
    print(f"[OK] File aggiornato salvato in: {output_path}")


# =========
# ESEMPIO
# =========

conf_path = "/content/NeuS_thesis/confs/long_test.conf"
base_exp_dir_value =  "/content/drive/MyDrive/Tesi/neus/Ablation_test/Experiments/02_SM/training" #"/content/drive/MyDrive/Tesi/neus/Ablation_test/Experiments/07_TO_80/training"
#
data_dir_value = "/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_simple/Mask_Segmentation/Pose_GT/CORTO_neus" #"/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_COLMAP/Merged_Orbit_80/Merged_Orbit_neus"
#
end_iter_value = 300000

update_conf_params(
    conf_path=conf_path,
    base_exp_dir_value=base_exp_dir_value,
    data_dir_value=data_dir_value,
    end_iter_value=end_iter_value
)

In [ ]:
# Run the multi-view geometry-proxy pipeline for the ablation case.
%cd /content/NeuS_thesis

!python custom_codes/uncertainty/accumulate_multiview_geometry_proxy.py \
  --conf ./confs/long_test.conf \
  --image_indices 0,7,13,20,26,33,40,46,53,59,66,73,79,86,92,99 \
  --rays_per_image 64 \
  --grid_resolution 32 \
  --seed 42 \
  --output_dir /content/drive/MyDrive/Tesi/neus/Ablation_test/bayesSDF/02_SM/multiview_geometry_16views

In [ ]:
# Display the sparsification curve for the ablation-case validation.
display(
    Image(
        filename="/content/drive/MyDrive/Tesi/neus/Ablation_test/bayesSDF/"
        "02_SM/mesh_validation/sparsification_curve.png"
    )
)